# 08a — Learning Mechanisms: Hebbian, Competitive, Boltzmann

Corresponds to: Note 08 — Learning Mechanisms: Hebbian, Competitive, Boltzmann

**Libraries introduced in this notebook:** No new libraries — numpy only (introduced in Note 03).

**What this notebook demonstrates:**
1. Hebbian learning — weight updates, co-occurrence detection, unbounded growth
2. Oja's rule — bounded weight growth, convergence to principal component direction
3. Competitive learning — winner-takes-all updates, prototype convergence
4. Boltzmann energy — computing state energy and probability from the energy function

---
## Part 1: Hebbian Learning

The Hebbian rule: Δwᵢⱼ = η × xᵢ × yⱼ

We reproduce the Exercise 08 Q10 trace exactly, then show unbounded growth when the same pattern repeats many times.

In [ ]:
import numpy as np

eta = 0.2
w = np.array([0.0, 0.0])
patterns = [np.array([1, 0]), np.array([1, 1]), np.array([1, 0])]
y = 1   # output fires for every pattern

print('Hebbian learning trace (Exercise 08 Q10):')
print(f'  Initial w = {w}')
for i, x in enumerate(patterns):
    dw = eta * x * y
    w  = w + dw
    print(f'  Pattern {i+1}: x={x}, y={y}  dw={dw}  w={w}')

print(f'\nFinal w = {w}')
print(f'w[0]={w[0]:.2f} > w[1]={w[1]:.2f}: neuron more selective for x1 (appeared twice)')

Now show unbounded growth: repeat pattern [1, 0] 50 times.

In [ ]:
w_grow = np.array([0.0, 0.0])
x = np.array([1, 0])

print('Unbounded growth — pattern [1,0] repeated 50 times:')
print(f'  {"Step":>5}   {"w[0]":>8}   {"w[1]":>8}')
print(f'  {"-"*5}   {"-"*8}   {"-"*8}')
for step in range(50):
    w_grow += eta * x * y
    if step < 5 or step % 10 == 9:
        print(f'  {step+1:>5}   {w_grow[0]:>8.2f}   {w_grow[1]:>8.2f}')

print()
print('w[0] grows without bound. w[1] stays 0 (x2=0 in this pattern).')
print('Pure Hebbian learning is unstable — weights must be bounded.')

---
## Part 2: Oja's Rule

Oja's rule: Δwᵢⱼ = η × yⱼ × (xᵢ − yⱼ × wᵢⱼ)

The decay term −yⱼ × wᵢⱼ prevents unbounded growth. We reproduce the Exercise 08 Q10(c) calculation, then run 50 steps to show convergence.

In [ ]:
# Exercise 08 Q10(c): one Oja step from w=[0.40, 0.20], x=[1,0]
w_oja = np.array([0.40, 0.20])
x     = np.array([1.0, 0.0])
eta   = 0.2

y_oja = w_oja @ x
dw    = eta * y_oja * (x - y_oja * w_oja)
w_new = w_oja + dw

print('Oja step (Exercise 08 Q10c):')
print(f'  w = {w_oja},  x = {x}')
print(f'  y = w.x = {y_oja:.4f}  (note: 0.4000)')
print(f'  dw = {dw.round(4)}  (note: [0.0672, -0.0064])')
print(f'  w_new = {w_new.round(4)}  (note: [0.4672, 0.1936])')
print(f'  norm before = {np.linalg.norm(w_oja):.4f},  after = {np.linalg.norm(w_new):.4f}')

Now run 100 Oja steps on pattern [1, 0] to show the weight norm stays bounded.

In [ ]:
w_oja2 = np.array([0.1, 0.0])   # small non-zero start -- Oja rule needs y = w.x != 0
x = np.array([1.0, 0.0])

print('Oja rule — 100 steps on pattern [1,0]:')
print(f'  {"Step":>5}   {"w[0]":>8}   {"w[1]":>8}   {"||w||": >8}')
print(f'  {"-"*5}   {"-"*8}   {"-"*8}   {"-"*8}')
for step in range(100):
    y_s = w_oja2 @ x
    w_oja2 += eta * y_s * (x - y_s * w_oja2)
    if step < 5 or step % 20 == 19:
        print(f'  {step+1:>5}   {w_oja2[0]:>8.4f}   {w_oja2[1]:>8.4f}   {np.linalg.norm(w_oja2):>8.4f}')

print()
print('w[0] converges to 1.0, w[1] stays 0. Norm converges to 1.0.')
print('Oja rule converges to the principal component direction — bounded, stable.')

---
## Part 3: Competitive Learning

Winner-takes-all: the neuron whose weight vector is closest to the input wins and updates toward the input. Losers do not update.

We use 3 neurons and 2D inputs drawn from 3 clusters. After training, each neuron's weight vector should converge to one cluster centroid.

In [ ]:
np.random.seed(0)

# 3 clusters of 2D points
c1 = np.random.randn(20, 2) + np.array([0.0, 0.0])
c2 = np.random.randn(20, 2) + np.array([4.0, 0.0])
c3 = np.random.randn(20, 2) + np.array([2.0, 3.5])
data = np.vstack([c1, c2, c3])
np.random.shuffle(data)

true_centroids = np.array([[0.0, 0.0], [4.0, 0.0], [2.0, 3.5]])

# 3 neurons, random initial weights
W = np.random.randn(3, 2) * 0.5 + np.array([[0.5, 0.5], [3.5, 0.5], [2.0, 3.0]])
eta_comp = 0.3

print('Initial neuron weight vectors:')
for i, w in enumerate(W):
    print(f'  Neuron {i+1}: {w.round(3)}')
print()

# Train for 5 passes through the data
for epoch in range(5):
    for x in data:
        dists  = np.linalg.norm(W - x, axis=1)
        winner = np.argmin(dists)
        W[winner] += eta_comp * (x - W[winner])

print('Learned weight vectors after 5 epochs:')
for i, w in enumerate(W):
    print(f'  Neuron {i+1}: {w.round(3)}')
print()
print('True cluster centroids:')
for i, c in enumerate(true_centroids):
    print(f'  Cluster {i+1}: {c}')
print()
print('Each neuron converged to approximately one cluster centroid.')

---
## Part 4: Boltzmann Energy

The energy of a Boltzmann machine state is:
E(v, h) = −Σᵢⱼ wᵢⱼ vᵢ hⱼ − Σᵢ aᵢ vᵢ − Σⱼ bⱼ hⱼ

We compute the energy and probability for several states of a small network (2 visible, 2 hidden units) to verify that low-energy states have higher probability.

In [ ]:
import math

# Small Boltzmann machine: 2 visible, 2 hidden
# W[i,j] = weight between visible i and hidden j
W_bm = np.array([[1.0, -0.5],
                 [0.5,  1.0]])
a = np.array([0.2, -0.1])   # visible biases
b = np.array([0.3,  0.1])   # hidden biases

def energy(v, h, W, a, b):
    return -(v @ W @ h) - (a @ v) - (b @ h)

# All 16 possible states (2 visible x 2 hidden, binary)
states = []
for v0 in [0,1]:
    for v1 in [0,1]:
        for h0 in [0,1]:
            for h1 in [0,1]:
                v = np.array([v0, v1], dtype=float)
                h = np.array([h0, h1], dtype=float)
                e = energy(v, h, W_bm, a, b)
                states.append((v0, v1, h0, h1, e))

# Compute partition function Z and probabilities
Z = sum(math.exp(-s[4]) for s in states)

print('All states, energies, and probabilities:')
print(f'  {"v":>6}   {"h":>6}   {"Energy":>8}   {"P(v,h)":>8}')
print(f'  {"-"*6}   {"-"*6}   {"-"*8}   {"-"*8}')
for v0, v1, h0, h1, e in sorted(states, key=lambda s: s[4]):
    prob = math.exp(-e) / Z
    print(f'  [{v0},{v1}]   [{h0},{h1}]   {e:>8.4f}   {prob:>8.4f}')

print(f'\nPartition function Z = {Z:.4f}')
print(f'Sum of all probabilities = {sum(math.exp(-s[4])/Z for s in states):.4f}  (should be 1.0)')
print()
print('Lower energy = higher probability. The network naturally favours low-energy states.')

---
## Summary

| What was demonstrated | Key result |
|---|---|
| Hebbian learning | Weights grow proportional to co-activation — w[0] grows faster when x1 appears more often |
| Hebbian instability | Weights grow without bound when the same pattern repeats |
| Oja's rule | Decay term keeps norm bounded — converges to principal component direction (w→[1,0]) |
| Competitive learning | Each neuron converges to one cluster centroid — discovers structure without labels |
| Boltzmann energy | Lower energy = higher probability; partition function Z normalises all states to sum to 1 |